# Geometric Morphometric Data Augmentation & Imbalance Classification
## Experiment 2: Pure Real-Data Balanced Subset (220-220-220)

### 📌 Study Overview & Experimental Objective
* **Dataset Size & Class Distribution**: Balanced Subsample of 220 Real Specimens per Class (Total N = 660, 100% Biological Data)
* **Feature Selection**: Geometric Morphometric Procrustes Coordinates excluding noisy/collinear landmarks (**LM11**, **LM21**, **LM22**) $\rightarrow$ **41 total features** (`GPA_X1`..`GPA_Y20`, `Centroid_Size`, `age_month`, `gender_num`).
* **Primary Metric Focus**: **SAM Recall**, **SAM Precision**, **SAM F1**, **Macro F1**, and **Balanced Accuracy** (avoiding raw overall accuracy which masks minor class performance).

---
### 🔄 Experimental Pipeline
```text
ONC (220 random) + MAM (220 random) + SAM (all 220) -> 5-Fold CV -> XGBoost -> Evaluation
```


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, balanced_accuracy_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings("ignore")

# Global Reproducibility Seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ Libraries loaded successfully.")


✅ Libraries loaded successfully.

In [ ]:
# Load Cleaned Geometric Morphometric Data
data_path = r"e:\ap-govt-EDA-model-tainning-testing\GM_out_of_sample_classification-main\GM_out_of_sample_classification-ap-govt\outliers_quality_of_data_extraction\all_experiments_results.csv"
df = pd.read_csv(data_path)

# Encode Gender
df['gender_num'] = df['gender'].map({'M': 0, 'F': 1, 'Male': 0, 'Female': 1}).fillna(0)

# Exclude Noisy & Collinear Landmarks (LM11, LM21, LM22)
exclude_lms = ['11', '21', '22']
exclude_cols = [f'GPA_X{lm}' for lm in exclude_lms] + [f'GPA_Y{lm}' for lm in exclude_lms]

feature_cols = [c for c in df.columns if c.startswith('GPA_') and c not in exclude_cols] + ['Centroid_Size', 'age_month', 'gender_num']
target_col = 'Class_Type'

print(f"📊 Cleaned Dataset Shape: {df.shape}")
print(f"🧬 Selected Feature Count: {len(feature_cols)} (Excluding LM11, LM21, LM22)")
print("🏷️ Full Dataset Class Distribution:")
print(df[target_col].value_counts())


📊 Cleaned Dataset Shape: (6207, 104)
🧬 Selected Feature Count: 41 (Excluding LM11, LM21, LM22)
🏷️ Full Dataset Class Distribution:
onc    5393
mam     722
sam     220
Name: Class_Type, dtype: int64

In [3]:
# Experiment 2 Setup: 220-220-220 Pure Real Subsample
df_sam = df[df['Class_Type'] == 'sam']
df_mam = df[df['Class_Type'] == 'mam'].sample(n=220, random_state=RANDOM_STATE)
df_onc = df[df['Class_Type'] == 'onc'].sample(n=220, random_state=RANDOM_STATE)

df_exp2 = pd.concat([df_onc, df_mam, df_sam]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

X_exp = df_exp2[feature_cols].values
y_exp = df_exp2['Class_Type'].map({'onc': 0, 'mam': 1, 'sam': 2}).values

print(f"Experiment 2 Subsample Matrix: {X_exp.shape}")
print(f"Class distribution: ONC=220, MAM=220, SAM=220 (100% Real Data, Clean Scientific Benchmark)")


Experiment 2 Subsample Matrix: (657, 41)
Class distribution: ONC=220, MAM=220, SAM=220 (100% Real Data, Clean Scientific Benchmark)

In [4]:
# Stratified 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

recalls, precisions, f1s_sam, macro_f1s, balanced_accs = [], [], [], [], []
cm_total = np.zeros((3, 3), dtype=int)
class_names = ['ONC', 'MAM', 'SAM']

print(f"🚀 Running 5-Fold Stratified Cross-Validation (Exp 2: Pure Real-Data Balanced Subset (220-220-220))...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_exp, y_exp), 1):
    X_tr, y_tr = X_exp[train_idx], y_exp[train_idx]
    X_te, y_te = X_exp[test_idx], y_exp[test_idx]
    
    # Apply Resampling / Weighting strictly INSIDE training fold to prevent data leakage
    if "exp2" == "exp1":
        sample_weights = compute_sample_weight('balanced', y_tr)
        X_res, y_res = X_tr, y_tr
    elif "exp2" == "exp2":
        X_res, y_res = X_tr, y_tr
    elif "exp2" == "exp3":
        bsmote = BorderlineSMOTE(sampling_strategy={2: 400}, random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_res = bsmote.fit_resample(X_tr, y_tr)
    elif "exp2" == "exp4":
        bsmote = BorderlineSMOTE(sampling_strategy={2: 577}, random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_res = bsmote.fit_resample(X_tr, y_tr)
    elif "exp2" == "exp5":
        smote_tomek = SMOTETomek(random_state=RANDOM_STATE)
        X_res, y_res = smote_tomek.fit_resample(X_tr, y_tr)
    elif "exp2" == "exp6":
        bsmote = BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=3)
        X_res, y_res = bsmote.fit_resample(X_tr, y_tr)

    # Instantiate XGBoost Model
    model = XGBClassifier(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4
    )
    
    if "exp2" == "exp1":
        model.fit(X_tr, y_tr, sample_weight=sample_weights)
    else:
        model.fit(X_res, y_res)
    
    # Evaluate on unseen test fold
    y_pred = model.predict(X_te)
    
    rec = recall_score(y_te, y_pred, labels=[2], average=None)[0]
    prec = precision_score(y_te, y_pred, labels=[2], average=None, zero_division=0)[0]
    f1_s = f1_score(y_te, y_pred, labels=[2], average=None)[0]
    m_f1 = f1_score(y_te, y_pred, average='macro')
    b_acc = balanced_accuracy_score(y_te, y_pred)
    
    recalls.append(rec)
    precisions.append(prec)
    f1s_sam.append(f1_s)
    macro_f1s.append(m_f1)
    balanced_accs.append(b_acc)
    cm_total += confusion_matrix(y_te, y_pred, labels=[0, 1, 2])
    
    print(f"Fold {fold}: SAM Recall={rec:.4f} | SAM Prec={prec:.4f} | SAM F1={f1_s:.4f} | Macro F1={m_f1:.4f} | Bal Acc={b_acc:.4f}")


🚀 Running 5-Fold Stratified Cross-Validation (Exp 2: Pure Real-Data Balanced Subset (220-220-220))...

Fold 1: SAM Recall=0.5000 | SAM Prec=0.5238 | SAM F1=0.5116 | Macro F1=0.5765 | Bal Acc=0.5758
Fold 2: SAM Recall=0.6136 | SAM Prec=0.6136 | SAM F1=0.6136 | Macro F1=0.6239 | Bal Acc=0.6288
Fold 3: SAM Recall=0.6279 | SAM Prec=0.5625 | SAM F1=0.5934 | Macro F1=0.6491 | Bal Acc=0.6487
Fold 4: SAM Recall=0.5581 | SAM Prec=0.5217 | SAM F1=0.5393 | Macro F1=0.5571 | Bal Acc=0.5573
Fold 5: SAM Recall=0.6279 | SAM Prec=0.6923 | SAM F1=0.6585 | Macro F1=0.5623 | Bal Acc=0.5578

In [5]:
# Compute Aggregated Metrics Across Folds
mean_sam_rec = np.mean(recalls)
mean_sam_prec = np.mean(precisions)
mean_sam_f1 = np.mean(f1s_sam)
mean_macro_f1 = np.mean(macro_f1s)
mean_bal_acc = np.mean(balanced_accs)

print("\n" + "="*65)
print(f"🏆 SUMMARY RESULTS: EXP 2 - PURE REAL-DATA BALANCED SUBSET (220-220-220)")
print("="*65)
print(f"🎯 SAM Recall        : {mean_sam_rec:.4f} ({mean_sam_rec*100:.2f}%)")
print(f"🎯 SAM Precision     : {mean_sam_prec:.4f} ({mean_sam_prec*100:.2f}%)")
print(f"🎯 SAM F1-Score      : {mean_sam_f1:.4f} ({mean_sam_f1*100:.2f}%)")
print(f"📈 Macro F1-Score    : {mean_macro_f1:.4f} ({mean_macro_f1*100:.2f}%)")
print(f"⚖️  Balanced Accuracy: {mean_bal_acc:.4f} ({mean_bal_acc*100:.2f}%)")
print("="*65)

# Plot Aggregated Confusion Matrix
plt.figure(figsize=(7, 5))
sns.heatmap(cm_total, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix (5-Fold Total) - Exp 2: Pure Real-Data Balanced Subset (220-220-220)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Class", fontsize=11)
plt.ylabel("True Class", fontsize=11)
plt.tight_layout()
plt.show()


🏆 SUMMARY RESULTS: EXP 2 - PURE REAL-DATA BALANCED SUBSET (220-220-220)
🎯 SAM Recall        : 0.5855 (58.55%)
🎯 SAM Precision     : 0.5828 (58.28%)
🎯 SAM F1-Score      : 0.5833 (58.33%)
📈 Macro F1-Score    : 0.5938 (59.38%)
⚖️  Balanced Accuracy: 0.5937 (59.37%)